# Save_Model.ipynb
This notebook rebuilds the **exact** preprocessing → TF-IDF → Linear SVM pipeline from the main WELFake project notebook, trains the final model, and saves it (as `.pkl` files via `pickle`) so they can be downloaded and used for deployment.

**Before running:** upload `WELFake_Dataset.csv` to the Colab file browser (or mount Drive and update the path in Cell 2), then run all cells top to bottom (`Runtime > Run all`).


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import pickle

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


## 2. NLTK resources
These are the exact resources the preprocessing pipeline needs (tokenizer, stopwords, lemmatizer).

In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


## 3. Load Dataset

In [ ]:
df = pd.read_csv("WELFake_Dataset.csv")
print(df.shape)
df.head()


## 4. Basic Cleaning
Same cleanup steps used in the main notebook: drop the stray index column (if present), drop duplicate rows, drop rows with no `text`, fill missing `title` with an empty string, and drop rows that end up with zero words.

In [ ]:
if "Unnamed: 0" in df.columns:
    df.drop("Unnamed: 0", axis=1, inplace=True)

df.drop_duplicates(inplace=True)
df = df.dropna(subset=['text'])
df["title"] = df["title"].fillna("")

df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df = df[df['word_count'] > 0]
df = df.drop(columns=['word_count'])

print(df.shape)


## 5. Combine Title + Text

In [ ]:
df["content"] = df["title"] + " " + df["text"]


## 6. Text Cleaning
Removes URLs, emails, HTML tags, newlines/tabs, and collapses extra whitespace — identical to the functions used in the main notebook.

In [ ]:
def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_emails(text):
    return re.sub(r'\S+@\S+', '', text)

def remove_html(text):
    return re.sub(r'<.*?>', '', text)

def remove_newlines(text):
    return text.replace('\n', ' ')

def remove_tabs(text):
    return text.replace('\t', ' ')

def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

def clean_text(text):
    text = remove_urls(text)
    text = remove_emails(text)
    text = remove_html(text)
    text = remove_newlines(text)
    text = remove_tabs(text)
    text = remove_extra_spaces(text)
    return text

df['clean_content'] = df['content'].apply(clean_text)


## 7. Normalization (Lowercase + Punctuation Removal)

In [ ]:
def to_lowercase(text):
    return text.lower()

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['normalized_content'] = df['clean_content'].apply(to_lowercase)
df['normalized_content'] = df['normalized_content'].apply(remove_punctuation)


## 8. Tokenization

In [ ]:
df['tokens'] = df['normalized_content'].apply(word_tokenize)


## 9. Stopword Removal

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['tokens'] = df['tokens'].apply(remove_stopwords)


## 10. Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df['lemmatized_tokens'] = df['tokens'].apply(lemmatize_tokens)


## 11. Train-Test Split

In [ ]:
X = df['lemmatized_tokens']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


## 12. Join Tokens Back into Strings
`TfidfVectorizer` expects strings, not token lists.

In [ ]:
X_train_text = X_train.apply(lambda x: " ".join(x))
X_test_text = X_test.apply(lambda x: " ".join(x))


## 13. TF-IDF Vectorization
Same parameters used in the main notebook's final TF-IDF experiments.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)


## 14. Train Final Model
Linear SVM with the best hyperparameters found during `RandomizedSearchCV` tuning in the main notebook (`C=1`, `loss='squared_hinge'`).

In [ ]:
model = LinearSVC(
    C=1,
    loss='squared_hinge',
    max_iter=5000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)


## 15. Sanity Check
Quick check to confirm this matches the ~95% test accuracy / ~94.7% F1 reported in the main notebook.

In [ ]:
y_test_pred = model.predict(X_test_tfidf)
y_test_score = model.decision_function(X_test_tfidf)

print(f"Test Accuracy  : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Test Precision : {precision_score(y_test, y_test_pred):.4f}")
print(f"Test Recall    : {recall_score(y_test, y_test_pred):.4f}")
print(f"Test F1 Score  : {f1_score(y_test, y_test_pred):.4f}")
print(f"Test ROC AUC   : {roc_auc_score(y_test, y_test_score):.4f}")


## 16. Save Model & Vectorizer (pickle)

In [ ]:
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("linear_svm.pkl", "wb") as f:
    pickle.dump(model, f)

print("Saved Successfully")


## 17. Download Files

In [ ]:
from google.colab import files

files.download("tfidf_vectorizer.pkl")
files.download("linear_svm.pkl")
